<a href="https://colab.research.google.com/github/RajeevRathore7055/AdvancedGenAIHackathon-Solution/blob/main/Assignment-3%20%3D%20AI-POWERED-RESUME-SCREENING-SYSTEM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🤖 AI Resume Screening System with Tracing
### Data Science Internship · February 2026 | GenAI Assignment · LangChain + LangSmith

---

## 📌 What This Notebook Does

- Builds an **AI-powered Resume Screening System** using LangChain and OpenAI
- Evaluates candidates against a job description using **Skill Extraction → Matching → Scoring → Explanation**
- Traces every pipeline step live using **LangSmith** for debugging and monitoring
- Demonstrates **Explainable AI** — every score comes with a human-readable reason

---

## 🔁 Pipeline Architecture

In [1]:
# INSTALLING THE LIBRARIES

!pip install langchain langchain-google-genai langsmith google-generativeai -q

In [2]:
import os
from google.colab import userdata

# Load secrets from Colab
gemini_api_key = userdata.get("GEMINI_API_KEY")
langsmith_api_key = userdata.get("LANGSMITH_API_KEY")

# Set environment variables (for libraries to use internally)
os.environ["GOOGLE_API_KEY"] = gemini_api_key
os.environ["LANGCHAIN_API_KEY"] = langsmith_api_key

# LangSmith settings (mandatory)
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "AI-Resume-Screening"

# Optional validation (safe)
if not gemini_api_key or not langsmith_api_key:
    raise ValueError("❌ Missing API keys in Colab Secrets")

print("✅ Secure setup complete (Gemini + LangSmith)")

✅ Secure setup complete (Gemini + LangSmith)


In [3]:
from langchain_core.prompts import PromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.output_parsers import StrOutputParser

print("✅ All libraries imported successfully!")

✅ All libraries imported successfully!


In [9]:
# Initialize Gemini Model
llm = ChatGoogleGenerativeAI(
    model="gemini-1.5-flash",   # Free & fast model
    temperature=0.3             # Consistent outputs
)

print("✅ Gemini LLM ready!")

✅ Gemini LLM ready!


In [10]:
# ✅ Strong Candidate
resume_strong = """
Name: Aarav Sharma
Email: aarav@email.com

Skills: Python, Machine Learning, Deep Learning, SQL, TensorFlow,
        Scikit-learn, Pandas, NumPy, Data Visualization, Git

Experience:
- Data Scientist at TechCorp (3 years)
  → Built ML models for customer churn prediction (92% accuracy)
  → Deployed models using Flask APIs on AWS
  → Led team of 3 junior analysts

Education: B.Tech in Computer Science, IIT Delhi

Tools: Jupyter, Docker, Git, AWS, Tableau
"""

# ⚠️ Average Candidate
resume_average = """
Name: Priya Mehta
Email: priya@email.com

Skills: Python, SQL, Excel, Basic Machine Learning, Pandas

Experience:
- Data Analyst Intern at ABC Pvt Ltd (6 months)
  → Worked on sales data cleaning and reporting
  → Created basic dashboards in Excel
  → Wrote simple SQL queries

Education: B.Sc in Statistics, Delhi University

Tools: Excel, Jupyter, Google Sheets
"""

# ❌ Weak Candidate
resume_weak = """
Name: Rohit Verma
Email: rohit@email.com

Skills: MS Word, PowerPoint, Basic Excel, Typing

Experience:
- Data Entry Operator at XYZ Office (1 year)
  → Entered data into spreadsheets manually
  → Filed documents

Education: 12th Pass

Tools: MS Office
"""

# 📋 Job Description
job_description = """
Role: Data Scientist

Required Skills:
- Python (must have)
- Machine Learning and Deep Learning
- SQL (must have)
- TensorFlow or PyTorch
- Data Visualization (Tableau or PowerBI)
- Model Deployment experience

Experience Required: Minimum 2 years in Data Science

Tools Required: Git, Docker, Jupyter, AWS or GCP
"""

print("✅ All 3 resumes and Job Description loaded!")

✅ All 3 resumes and Job Description loaded!


In [11]:
# ---- PROMPT ----
extraction_prompt = PromptTemplate(
    input_variables=["resume"],
    template="""
You are an expert HR assistant.
Extract information ONLY from the resume below.
Do NOT assume or add any skills not mentioned in the resume.

Resume:
{resume}

Extract and return EXACTLY in this format:
Skills: <comma separated list>
Experience: <brief summary>
Tools: <comma separated list>
"""
)

# ---- CHAIN ----
extraction_chain = extraction_prompt | llm | StrOutputParser()

# ---- TEST ----
test_extract = extraction_chain.invoke({"resume": resume_strong})
print("=== EXTRACTION TEST (Strong) ===")
print(test_extract)

ChatGoogleGenerativeAIError: Error calling model 'gemini-1.5-flash' (NOT_FOUND): 404 NOT_FOUND. {'error': {'code': 404, 'message': 'models/gemini-1.5-flash is not found for API version v1beta, or is not supported for generateContent. Call ListModels to see the list of available models and their supported methods.', 'status': 'NOT_FOUND'}}

In [ ]:
# ---- PROMPT ----
matching_prompt = PromptTemplate(
    input_variables=["extracted_info", "job_description"],
    template="""
You are a recruiter comparing a candidate profile with a job description.

Candidate Profile:
{extracted_info}

Job Description:
{job_description}

Analyze and return EXACTLY in this format:
Matched Skills: <skills present in both candidate and job>
Missing Skills: <skills required in job but absent in candidate>
Experience Match: <Yes / Partial / No — with one line reason>
"""
)

# ---- CHAIN ----
matching_chain = matching_prompt | llm | StrOutputParser()

# ---- TEST ----
test_match = matching_chain.invoke({
    "extracted_info": test_extract,
    "job_description": job_description
})
print("=== MATCHING TEST (Strong) ===")
print(test_match)

In [ ]:
# ---- PROMPT ----
scoring_prompt = PromptTemplate(
    input_variables=["matched_info", "job_description"],
    template="""
You are an AI scoring system for recruitment.

Based on the match analysis and job description below,
assign a score between 0 and 100.

Match Analysis:
{matched_info}

Job Description:
{job_description}

Scoring Rules:
- 80 to 100 → Strong match (most required skills present)
- 50 to 79  → Average match (some skills present)
- 0  to 49  → Weak match (few or no relevant skills)

Return ONLY in this format:
Score: <number 0-100>
Category: <Strong / Average / Weak>
"""
)

# ---- CHAIN ----
scoring_chain = scoring_prompt | llm | StrOutputParser()

# ---- TEST ----
test_score = scoring_chain.invoke({
    "matched_info": test_match,
    "job_description": job_description
})
print("=== SCORING TEST (Strong) ===")
print(test_score)

In [ ]:
# ---- PROMPT ----
explanation_prompt = PromptTemplate(
    input_variables=["score", "matched_info"],
    template="""
You are an AI hiring assistant writing feedback for a recruiter.

Based on the score and match analysis below,
write a short professional explanation (3 to 5 sentences).
Explain clearly WHY this candidate received this score.

Score:
{score}

Match Analysis:
{matched_info}

Write the explanation below:
"""
)

# ---- CHAIN ----
explanation_chain = explanation_prompt | llm | StrOutputParser()

# ---- TEST ----
test_explanation = explanation_chain.invoke({
    "score": test_score,
    "matched_info": test_match
})
print("=== EXPLANATION TEST (Strong) ===")
print(test_explanation)

In [ ]:
def screen_resume(resume, job_desc, candidate_name):
    """
    Full pipeline:
    Resume → Extract → Match → Score → Explain
    """

    print(f"\n{'='*55}")
    print(f"  🔍 Processing Candidate: {candidate_name}")
    print(f"{'='*55}")

    # ── STEP 1: Skill Extraction ──
    print("\n📋 STEP 1: Extracting Skills...")
    extracted = extraction_chain.invoke({"resume": resume})
    print(extracted)

    # ── STEP 2: Matching ──
    print("\n🔗 STEP 2: Matching with Job Description...")
    matched = matching_chain.invoke({
        "extracted_info": extracted,
        "job_description": job_desc
    })
    print(matched)

    # ── STEP 3: Scoring ──
    print("\n🎯 STEP 3: Calculating Score...")
    score = scoring_chain.invoke({
        "matched_info": matched,
        "job_description": job_desc
    })
    print(score)

    # ── STEP 4: Explanation ──
    print("\n💬 STEP 4: Generating Explanation...")
    explanation = explanation_chain.invoke({
        "score": score,
        "matched_info": matched
    })
    print(explanation)

    print(f"\n✅ Done: {candidate_name}")

    return {
        "candidate"  : candidate_name,
        "extracted"  : extracted,
        "matched"    : matched,
        "score"      : score,
        "explanation": explanation
    }

print("✅ Pipeline function ready!")

In [ ]:
# ✅ Run Strong Candidate
result_strong = screen_resume(
    resume_strong,
    job_description,
    "Aarav Sharma (Strong)"
)

# ✅ Run Average Candidate
result_average = screen_resume(
    resume_average,
    job_description,
    "Priya Mehta (Average)"
)

# ✅ Run Weak Candidate
result_weak = screen_resume(
    resume_weak,
    job_description,
    "Rohit Verma (Weak)"
)

print("\n\n✅ All 3 candidates screened successfully!")
print("📊 Check traces at → https://smith.langchain.com")

In [ ]:
print("\n")
print("=" * 60)
print("        📊 FINAL RESUME SCREENING SUMMARY")
print("=" * 60)

results = [result_strong, result_average, result_weak]

for r in results:
    print(f"\n👤 Candidate  : {r['candidate']}")
    print(f"🎯 Score      : {r['score'].strip()}")
    print(f"💬 Explanation: {r['explanation'][:120].strip()}...")
    print("-" * 60)

print("\n🚀 Pipeline Complete! All steps traced in LangSmith.")

Cell 1  → Libraries install

Cell 2  → Keys activate + LangSmith
turns ON

Cell 3  → Imports load

Cell 4  → Gemini model initializes

Cell 5  → 3 resumes + JD ready

Cell 6  → Extraction chain built + tested

Cell 7  → Matching chain built + tested

Cell 8  → Scoring chain built + tested

Cell 9  → Explanation chain built + tested

Cell 10 → Full pipeline function defined

Cell 11 → All 3 candidates processed ← MAIN RUN

Cell 12 → Clean summary printed